# 03 · Produccion multifuente (N bases) — nivel AVANZADO

**Que hace**: consolida **2, 3, 4 o mas fuentes** en un registro unico empresarial:
deduplica dentro de cada fuente y cruza entre fuentes en una sola corrida
(`linkage`, el mismo Orchestrator del flujo de produccion), con fuentes confiables
(`trusted`), refinador multicampo opcional y **manifiesto de trazabilidad** en JSON.

**Disenado para** corridas grandes en Colab (checkpoints en `work_dir` sobre Drive:
si la sesion se cae, la re-ejecucion reutiliza las fases ya calculadas).

**Pasos**: Celda A (entorno) → Celda A2 (definiciones) → **Celda B** (config + correr).

> Motor: `rues-linker 0.11.0`. Salidas: GOLDEN, CORRELATIVA, MATRIZ_FUENTES,
> RESUMEN + `manifiesto_corrida.json`.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CELDA A — ENTORNO  (ejecutar UNA vez por sesion; no requiere edicion)
# ══════════════════════════════════════════════════════════════════════════
from __future__ import annotations

import importlib
import subprocess
import sys

VERSION_MINIMA = "0.11.0"
ORIGEN_GIT = "rues-linker @ git+https://github.com/EnriqueForero/rues-linker@v0.11.0"
RUTA_PAQUETE_DRIVE = "/content/drive/MyDrive/ProColombia/record_linkage"   # respaldo si no hay salida a GitHub

EN_COLAB = "google.colab" in sys.modules or importlib.util.find_spec("google.colab") is not None
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")


def _version_ok(v: str, minima: str = VERSION_MINIMA) -> bool:
    """True si la version instalada cumple la minima (comparacion numerica)."""
    def _t(x: str) -> tuple[int, ...]:
        base = x.split("+")[0].split("-")[0]
        return tuple(int(p) for p in base.split(".")[:3] if p.isdigit())
    try:
        return _t(v) >= _t(minima)
    except Exception:
        return False


def preparar_entorno() -> None:
    """Garantiza rues-linker >= 0.11.0: usa la instalada, o instala de GitHub,
    o (respaldo) en modo editable desde Drive. Falla con mensaje accionable."""
    try:
        import record_linkage
        if _version_ok(record_linkage.__version__):
            print(f"rues-linker {record_linkage.__version__} ya instalado — listo")
            return
        print(f"rues-linker {record_linkage.__version__} es anterior a {VERSION_MINIMA}; actualizando...")
    except ImportError:
        print("rues-linker no instalado; instalando desde GitHub...")
    intentos = [
        [sys.executable, "-m", "pip", "install", "-q", ORIGEN_GIT],
        [sys.executable, "-m", "pip", "install", "-q", "-e", RUTA_PAQUETE_DRIVE],
    ]
    for cmd in intentos:
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode == 0:
            break
    else:
        raise RuntimeError(
            "No fue posible instalar rues-linker.\n"
            f"  1) Verifique salida a internet para: https://github.com/EnriqueForero/rues-linker\n"
            f"  2) O que el paquete exista en Drive: {RUTA_PAQUETE_DRIVE}\n"
            f"  Detalle pip: {r.stderr.strip()[-400:]}"
        )
    importlib.invalidate_caches()
    import record_linkage
    importlib.reload(record_linkage)
    if not _version_ok(record_linkage.__version__):
        raise RuntimeError(
            f"Se instalo {record_linkage.__version__} pero se requiere >= {VERSION_MINIMA}. "
            "Reinicie el entorno (Entorno de ejecucion -> Reiniciar) y vuelva a correr esta celda."
        )
    print(f"rues-linker {record_linkage.__version__} instalado — listo")


preparar_entorno()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CELDA A2 — EXTRAS (definiciones; ejecutar una vez, no requiere edicion)
# ══════════════════════════════════════════════════════════════════════════

# ── Lectura y validacion de bases (fail-fast con mensajes accionables) ──────

import unicodedata
from pathlib import Path

import pandas as pd


def _norm(texto: str) -> str:
    """Normaliza un nombre de columna: sin tildes, mayusculas, sin espacios extremos."""
    t = unicodedata.normalize("NFKD", str(texto))
    return "".join(ch for ch in t if not unicodedata.combining(ch)).strip().upper()


def leer_base(
    ruta: str | Path,
    hoja: str | int = 0,
    fila_encabezado: int = 0,
    columnas_texto: tuple[str, ...] = (),
) -> pd.DataFrame:
    """Lee CSV/XLSX validando existencia y devolviendo columnas normalizadas.

    Args:
        ruta: ruta al archivo .csv, .xlsx o .xls (Drive o local).
        hoja: nombre o indice de hoja (solo Excel).
        fila_encabezado: fila (0-index) donde estan los encabezados.
        columnas_texto: columnas a forzar como texto (p. ej. NIT, para no
            perder ceros a la izquierda ni convertir a notacion cientifica).

    Returns:
        DataFrame con encabezados normalizados (sin tildes, en mayusculas).

    Raises:
        FileNotFoundError: si la ruta no existe (con la carpeta que si existe).
        ValueError: si la extension no es soportada.
    """
    p = Path(ruta)
    if not p.exists():
        padre = next((x for x in p.parents if x.exists()), Path("/"))
        vistos = ", ".join(sorted(f.name for f in padre.iterdir())[:12]) or "(vacia)"
        raise FileNotFoundError(
            f"No existe: {p}\n  La carpeta mas cercana que SI existe es: {padre}\n"
            f"  Contiene: {vistos}\n  Corrija la ruta o el nombre del archivo (copielo exacto)."
        )
    dtype = {c: "string" for c in columnas_texto}
    ext = p.suffix.lower()
    if ext == ".csv":
        df = pd.read_csv(p, header=fila_encabezado, dtype=dtype, encoding_errors="replace")
    elif ext in (".xlsx", ".xls", ".xlsm"):
        df = pd.read_excel(p, sheet_name=hoja, header=fila_encabezado, dtype=dtype)
    else:
        raise ValueError(f"Extension no soportada: '{ext}'. Use .csv o .xlsx")
    df.columns = [_norm(c) for c in df.columns]
    return df


def exigir_columnas(df: pd.DataFrame, requeridas: dict[str, str], contexto: str) -> pd.DataFrame:
    """Renombra columnas del usuario al contrato NIT/RAZON_SOCIAL/... y valida.

    Args:
        df: base leida.
        requeridas: {nombre_en_su_archivo: nombre_contrato}; los nombres del
            archivo se comparan ya normalizados (sin tildes, mayusculas).
        contexto: etiqueta para mensajes de error (p. ej. "BASE_A").

    Returns:
        DataFrame solo con las columnas del contrato, renombradas.
    """
    mapa = {_norm(k): v for k, v in requeridas.items()}
    faltan = [k for k in mapa if k not in df.columns]
    if faltan:
        raise KeyError(
            f"[{contexto}] No se hallaron las columnas {faltan}.\n"
            f"  Columnas disponibles: {list(df.columns)}\n"
            "  Corrija los nombres en la CONFIGURACION (se comparan sin tildes ni mayusculas)."
        )
    out = df[list(mapa)].rename(columns=mapa).copy()
    for c in out.columns:
        out[c] = out[c].astype("string").fillna("").str.strip()
    return out


# ── Exportacion (Excel hasta ~1M filas por hoja; CSV comprimido si excede) ──

LIMITE_EXCEL = 1_000_000


def exportar_resultados(
    tablas: dict[str, pd.DataFrame],
    carpeta: str | Path,
    nombre_base: str,
) -> list[Path]:
    """Exporta cada tabla; una sola .xlsx multi-hoja si todas caben, si no CSV.gz.

    Returns:
        Rutas de los archivos escritos.
    """
    carpeta = Path(carpeta)
    carpeta.mkdir(parents=True, exist_ok=True)
    escritos: list[Path] = []
    if all(len(t) <= LIMITE_EXCEL for t in tablas.values()):
        ruta = carpeta / f"{nombre_base}.xlsx"
        with pd.ExcelWriter(ruta, engine="openpyxl") as xw:
            for hoja, t in tablas.items():
                t.to_excel(xw, sheet_name=hoja[:31], index=False)
        escritos.append(ruta)
    else:
        for hoja, t in tablas.items():
            ruta = carpeta / f"{nombre_base}__{hoja}.csv.gz"
            t.to_csv(ruta, index=False, compression="gzip")
            escritos.append(ruta)
    for r in escritos:
        print(f"  Archivo escrito: {r}")
    return escritos


import contextlib
import gc
import hashlib
import io
import json
import time
from dataclasses import dataclass, field
from datetime import datetime, timezone

from record_linkage import __version__ as _VERSION_LIB
from record_linkage import default_colombia_profile, linkage


@dataclass(frozen=True)
class EspecBase:
    """Una fuente de entrada y el mapeo de SUS columnas al contrato del motor."""
    nombre: str
    archivo: str
    col_nit: str
    col_nombre: str
    col_ciudad: str | None = None
    hoja: str | int = 0
    fila_encabezado: int = 0

    def __post_init__(self) -> None:
        if not self.nombre.strip():
            raise ValueError("EspecBase.nombre no puede estar vacio")

    def cargar(self) -> pd.DataFrame:
        crudo = leer_base(self.archivo, self.hoja, self.fila_encabezado, (self.col_nit,))
        mapa = {self.col_nit: "NIT", self.col_nombre: "RAZON_SOCIAL"}
        if self.col_ciudad:
            mapa[self.col_ciudad] = "CIUDAD"
        df = exigir_columnas(crudo, mapa, self.nombre)
        mb = df.memory_usage(deep=True).sum() / 1e6
        print(f"[{self.nombre}] {len(df):,} registros  (~{mb:.0f} MB)")
        return df


@dataclass
class ConfigProduccion:
    """Configuracion completa de la corrida multifuente (validada al construir).

    Attributes:
        etiqueta: identifica la corrida (carpeta de salida y manifiesto).
        fuentes: 2 o mas EspecBase.
        confiables: nombres de fuentes ya depuradas (subset de fuentes).
        perfil: perfil del Orchestrator (produccion_estandar / produccion_exhaustiva
            / alta_precision).
        usar_ciudad: activa CIUDAD si TODAS las fuentes la declaran.
        refinador_colombia: activa el MatchingProfile calibrado para Colombia
            (post-proceso que separa falsos positivos) + auditoria de decisiones.
        dir_salida: carpeta raiz de salidas; work_dir de checkpoints vive dentro.
    """
    etiqueta: str
    fuentes: list[EspecBase]
    confiables: set[str] = field(default_factory=set)
    perfil: str = "produccion_estandar"
    usar_ciudad: bool = True
    refinador_colombia: bool = False
    dir_salida: str = "/content/drive/MyDrive/ProColombia/Deduplicacion/salidas"

    def __post_init__(self) -> None:
        if len(self.fuentes) < 2:
            raise ValueError("Se requieren al menos 2 fuentes (para 1 use el notebook 01)")
        nombres = [f.nombre for f in self.fuentes]
        if len(set(nombres)) != len(nombres):
            raise ValueError(f"Nombres de fuente repetidos: {nombres}")
        sobran = self.confiables - set(nombres)
        if sobran:
            raise ValueError(f"'confiables' contiene fuentes no declaradas: {sobran}")
        if not self.etiqueta.strip():
            raise ValueError("etiqueta no puede estar vacia")

    @property
    def carpeta_corrida(self) -> Path:
        return Path(self.dir_salida) / self.etiqueta


def _huella(df: pd.DataFrame) -> str:
    """Huella corta y determinista del insumo (trazabilidad del manifiesto)."""
    h = hashlib.sha256()
    h.update(f"{df.shape}|{list(df.columns)}".encode())
    h.update(pd.util.hash_pandas_object(df.head(1000), index=False).values.tobytes())
    return h.hexdigest()[:16]


def ejecutar_produccion(cfg: ConfigProduccion) -> dict[str, pd.DataFrame]:
    """Corrida completa: cargar N fuentes -> linkage() -> matriz -> manifiesto -> export."""
    t0 = time.time()
    fuentes = {f.nombre: f.cargar() for f in cfg.fuentes}
    con_ciudad = cfg.usar_ciudad and all("CIUDAD" in d.columns for d in fuentes.values())
    if cfg.usar_ciudad and not con_ciudad:
        print("Aviso: no todas las fuentes declaran ciudad; se corre SIN esa variable.")

    ruido = io.StringIO()
    with contextlib.redirect_stdout(ruido), contextlib.redirect_stderr(ruido):
        res = linkage(
            fuentes,
            trusted_sources=cfg.confiables or None,
            col_ciudad="CIUDAD" if con_ciudad else None,
            profile=cfg.perfil,
            matching_profile=default_colombia_profile() if cfg.refinador_colombia else None,
            return_matcher_audit=cfg.refinador_colombia,
            work_dir=str(cfg.carpeta_corrida / "trabajo"),
        )

    golden, correlativa = res["golden"], res["correlative"]
    presencia = (
        correlativa.assign(_uno=1)
        .pivot_table(index="ID_GRUPO", columns="SRC", values="_uno", aggfunc="max", fill_value=0)
        .reset_index()
    )
    resumen = pd.DataFrame(
        {
            "indicador": (
                ["registros_totales", "grupos_unicos", "perfil", "confiables",
                 "ciudad_usada", "refinador_colombia", "duracion_seg"]
                + [f"registros_{n}" for n in fuentes]
            ),
            "valor": (
                [len(correlativa), correlativa["ID_GRUPO"].nunique(), cfg.perfil,
                 ",".join(sorted(cfg.confiables)) or "-", con_ciudad,
                 cfg.refinador_colombia, round(time.time() - t0, 1)]
                + [len(d) for d in fuentes.values()]
            ),
        }
    )
    tablas = {"GOLDEN": golden, "CORRELATIVA": correlativa,
              "MATRIZ_FUENTES": presencia, "RESUMEN": resumen}
    if cfg.refinador_colombia and "matcher_decisions" in res:
        tablas["AUDITORIA_REFINADOR"] = res["matcher_decisions"]

    manifiesto = {
        "corrida": cfg.etiqueta,
        "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "version_rues_linker": _VERSION_LIB,
        "perfil": cfg.perfil,
        "confiables": sorted(cfg.confiables),
        "ciudad_usada": con_ciudad,
        "refinador_colombia": cfg.refinador_colombia,
        "fuentes": {n: {"filas": len(d), "huella": _huella(d)} for n, d in fuentes.items()},
        "resultados": {"grupos": int(correlativa["ID_GRUPO"].nunique()),
                        "registros": int(len(correlativa))},
    }
    cfg.carpeta_corrida.mkdir(parents=True, exist_ok=True)
    ruta_man = cfg.carpeta_corrida / "manifiesto_corrida.json"
    ruta_man.write_text(json.dumps(manifiesto, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"  Manifiesto: {ruta_man}")

    exportar_resultados(tablas, cfg.carpeta_corrida, f"consolidado_{cfg.etiqueta}")
    print(f"\nListo: {manifiesto['resultados']['grupos']:,} empresas unicas "
          f"a partir de {manifiesto['resultados']['registros']:,} registros "
          f"en {time.time() - t0:,.0f} s")
    del fuentes
    gc.collect()
    return tablas

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CELDA B — CONFIGURAR Y EJECUTAR        (LA UNICA CELDA QUE USTED EDITA)
# ══════════════════════════════════════════════════════════════════════════
RAIZ = "/content/drive/MyDrive/ProColombia/Deduplicacion/datos/2026-07"

CONFIG = ConfigProduccion(
    etiqueta="CONSOLIDADO_2026_07",
    fuentes=[
        EspecBase("RUES",     f"{RAIZ}/rues.xlsx",     col_nit="NIT",  col_nombre="RAZON SOCIAL", col_ciudad="CIUDAD"),
        EspecBase("CRM",      f"{RAIZ}/crm.xlsx",      col_nit="NIT",  col_nombre="NOMBRE",       col_ciudad="CIUDAD"),
        EspecBase("EVENTOS",  f"{RAIZ}/eventos.csv",   col_nit="NIT",  col_nombre="EMPRESA",      col_ciudad=None),
        EspecBase("GAZELLE",  f"{RAIZ}/gazelle.xlsx",  col_nit="NIT9", col_nombre="COMPANY",      col_ciudad=None),
    ],
    confiables={"RUES"},
    perfil="produccion_estandar",      # produccion_exhaustiva | alta_precision
    refinador_colombia=False,           # True: post-proceso calibrado + AUDITORIA
)

tablas = ejecutar_produccion(CONFIG)
tablas["RESUMEN"]

### Operacion y reanudacion

- **Checkpoints**: cada fase del pipeline queda en `.../<etiqueta>/trabajo/`. Si Colab
  se reinicia a mitad de una corrida grande, vuelva a ejecutar la Celda B con la MISMA
  `etiqueta`: las fases completas se reutilizan.
- **Corrida nueva sobre datos nuevos**: cambie la `etiqueta` (o borre `trabajo/`).
- **MATRIZ_FUENTES**: por cada empresa unica, en que fuentes aparece (1/0) — ideal
  para responder "¿cuantas del CRM estan en RUES?" con una tabla dinamica.
- **Refinador Colombia**: si observa grupos "pegados" que no deberian (nombres genericos),
  active `refinador_colombia=True` y revise la hoja `AUDITORIA_REFINADOR` con la decision
  par a par.
- **Memoria**: para >2M de registros use Colab con RAM alta si es posible y evite tener
  los Excel de entrada abiertos en otras pestanas de Drive.